# STEP 2: Apply the Pre-Trained Model to the Dataset

In this step we are taking the pre-trained spacy model with the 'en_core_web_sm' model to the entire dataset and see how the model performs in terms of its ability to correctly categorize specific tokens within the entities that are associated with the spacy model. We also are trying to see how it classifies when using the custom labels that we created that are relevant to the domain of the data. 

In [108]:
# ============================================================
# SETUP: Install and import spaCy
# ============================================================
# spaCy is pre-installed in Colab. If running locally:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from spacy import displacy
import json
import pandas as pd
from collections import Counter
import re
import pandas as pd

# Load the small English model
# This model was trained on web text (blogs, news, comments)
# It recognizes entities like PERSON, ORG, GPE, DATE, MONEY, etc.
nlp = spacy.load("en_core_web_sm")

print(f"Model loaded: {nlp.meta['name']}")
print(f"Pipeline components: {nlp.pipe_names}")

Model loaded: core_web_sm
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [109]:
#Creates an empty array to store the information from the Dataset
task = []

#Opens the datafile and loads it line by line into the task list
with open(r"C:\Users\ysass\OneDrive\Desktop\Grad School Files\BSAN 6200 Text Mining and Social Media Analytics\text-analytics-a4-Sasson-Acosta\text-analytics-a4-Sasson-Acosta\data\raw\dataset_train.json", encoding="utf-8") as f:
    for line in f:
        task.append(json.loads(line))

In [110]:
# ============================================================
# SETUP: Install and import spaCy
# ============================================================
# spaCy is pre-installed in Colab. If running locally:
# !pip install spacy
# !python -m spacy download en_core_web_sm



# Load the small English model
# This model was trained on web text (blogs, news, comments)
# It recognizes entities like PERSON, ORG, GPE, DATE, MONEY, etc.
nlp = spacy.load("en_core_web_sm")

print(f"Model loaded: {nlp.meta['name']}")
print(f"Pipeline components: {nlp.pipe_names}")

Model loaded: core_web_sm
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [111]:
#Empty list to store the reviews
text_list=[]

#Takes each review and loads it by joing the words in each review and stores it as an entry in the list
for tasks in task:
    text = tasks["tokens"]
    text=" ".join(text)
    text_list.append(text)

In [ ]:
text_list

In [ ]:
# ============================================================
# BASIC NER: Process a single sentence
# ============================================================
# nlp() processes text and returns a Doc object with all annotations

for text in text_list:
    doc = nlp(text)
    
    # Loop through detected entities
    # Each entity has:
    #   .text       → the actual text span (e.g., "Thomas Jefferson")
    #   .start_char → character position where entity starts
    #   .end_char   → character position where entity ends
    #   .label_     → entity type label (e.g., "PERSON")
    
    print(f"Text: {text}")
    print(f"{'Entity':<30} {'Start':>5} {'End':>5}  {'Label'}")
    print("-" * 60)
    for ent in doc.ents:
        print(f"{ent.text:<30} {ent.start_char:>5} {ent.end_char:>5}  {ent.label_}")

In [ ]:
#Takes each review and utilizes it 
for text in text_list:
    doc = nlp(text)
    print(f"\nText: {text}")

    # Render the entity visualization
    displacy.render(doc, style="ent", jupyter=True, options={'distance': 120})

    # Also print entities in structured format
    for ent in doc.ents:
        print(f"  → {ent.text:<25} {ent.label_:<10} ({ent.start_char}-{ent.end_char})")

    if not doc.ents:
        print("  → No entities detected!")

In [115]:
# update this path only if needed
label_path = r"C:\Users\ysass\OneDrive\Desktop\Grad School Files\BSAN 6200 Text Mining and Social Media Analytics\text-analytics-a4-Sasson-Acosta\text-analytics-a4-Sasson-Acosta\data\raw\dataset_label.json"

with open(label_path, "r", encoding="utf-8") as f:
    label_map = json.load(f)

id_to_label = {v: k for k, v in label_map.items()}
id_to_label

{0: 'O',
 1: 'B-Rating',
 2: 'I-Rating',
 3: 'B-Amenity',
 4: 'I-Amenity',
 5: 'B-Location',
 6: 'I-Location',
 7: 'B-Restaurant_Name',
 8: 'I-Restaurant_Name',
 9: 'B-Price',
 10: 'B-Hours',
 11: 'I-Hours',
 12: 'B-Dish',
 13: 'I-Dish',
 14: 'B-Cuisine',
 15: 'I-Price',
 16: 'I-Cuisine'}

In [116]:
#Standarize to fit our custom tags to the SPACY model
SPACY_TO_RESTAURANT = {
    "ORG": "Restaurant_Name",
    "FAC": "Restaurant_Name",
    "GPE": "Location",
    "LOC": "Location",
    "NORP": "Cuisine",
    "TIME": "Hours",
    "DATE": "Hours",
    "MONEY": "Price",
    "CARDINAL": "Rating",
    "ORDINAL": "Rating"
}

#Get rid of the bio elements in the tags
def strip_bio(label):
    if label == "O":
        return "O"
    return label.replace("B-", "").replace("I-", "")

#Determine the positions of the entities
def tokens_to_text_and_offsets(tokens):
    text = " ".join(tokens)
    offsets = []
    current = 0
    
    for token in tokens:
        start = current
        end = start + len(token)
        offsets.append((start, end))
        current = end + 1
    
    return text, offsets

#Takes the entities from the review and converts the tags to entity spans with the tag, token, and label combining the three into one string.
def gold_entities_from_tokens(tokens, tag_ids, id_to_label):
    text, offsets = tokens_to_text_and_offsets(tokens)
    entities = []

    current_label = None
    current_start = None
    current_end = None

    for i, tag_id in enumerate(tag_ids):
        raw_label = id_to_label[tag_id]
        start, end = offsets[i]

        if raw_label == "O":
            if current_label is not None:
                entities.append(
                    (current_start, current_end, text[current_start:current_end], current_label)
                )
                current_label = None
                current_start = None
                current_end = None
            continue

        prefix = raw_label.split("-")[0]
        clean_label = strip_bio(raw_label)

        if prefix == "B":
            if current_label is not None:
                entities.append(
                    (current_start, current_end, text[current_start:current_end], current_label)
                )
            current_label = clean_label
            current_start = start
            current_end = end

        elif prefix == "I":
            if current_label == clean_label:
                current_end = end
            else:
                if current_label is not None:
                    entities.append(
                        (current_start, current_end, text[current_start:current_end], current_label)
                    )
                current_label = clean_label
                current_start = start
                current_end = end

    if current_label is not None:
        entities.append(
            (current_start, current_end, text[current_start:current_end], current_label)
        )

    return text, entities

#Map spacy labels to the custom labels
def spacy_entities_mapped(text, nlp, label_mapping):
    """
    Convert spaCy labels into your restaurant-domain labels.
    Unmapped labels are ignored.
    """
    doc = nlp(text)
    entities = []
    
    for ent in doc.ents:
        mapped_label = label_mapping.get(ent.label_)
        if mapped_label is not None:
            entities.append((ent.start_char, ent.end_char, ent.text, mapped_label))
    
    return entities

#Compare the annotated to the predicted entities
def compare_gold_vs_pred(gold_ents, pred_ents):
    gold_exact = {(s, e, label): txt for s, e, txt, label in gold_ents}
    pred_exact = {(s, e, label): txt for s, e, txt, label in pred_ents}

    gold_by_span = {(s, e): (txt, label) for s, e, txt, label in gold_ents}
    pred_by_span = {(s, e): (txt, label) for s, e, txt, label in pred_ents}

    tp = []
    fp = []
    fn = []
    wrong = []

    for key in pred_exact:
        if key in gold_exact:
            s, e, label = key
            tp.append((s, e, pred_exact[key], label))

    wrong_spans = set()
    for span, (pred_text, pred_label) in pred_by_span.items():
        if span in gold_by_span:
            gold_text, gold_label = gold_by_span[span]
            if pred_label != gold_label:
                wrong.append({
                    "start": span[0],
                    "end": span[1],
                    "text": pred_text,
                    "gold_label": gold_label,
                    "pred_label": pred_label
                })
                wrong_spans.add(span)

    for s, e, txt, label in pred_ents:
        if (s, e, label) not in gold_exact and (s, e) not in wrong_spans:
            fp.append((s, e, txt, label))

    for s, e, txt, label in gold_ents:
        if (s, e, label) not in pred_exact and (s, e) not in wrong_spans:
            fn.append((s, e, txt, label))

    return tp, fp, fn, wrong

#Calculate Error Analysis
def calc_metrics(tp, fp, fn, wrong):
    tp_n = len(tp)
    fp_n = len(fp) + len(wrong)
    fn_n = len(fn) + len(wrong)

    precision = tp_n / (tp_n + fp_n) if (tp_n + fp_n) > 0 else 0
    recall = tp_n / (tp_n + fn_n) if (tp_n + fn_n) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return tp_n, fp_n, fn_n, precision, recall, f1

In [117]:
all_tp = []
all_fp = []
all_fn = []
all_wrong = []
error_examples = []

#Take the tokens and reviews standardize them compare them to what the official results are and add them to the relevant 
for item in task:
    tokens = item["tokens"]
    tag_ids = item["tags"]

    text, gold_ents = gold_entities_from_tokens(tokens, tag_ids, id_to_label)
    pred_ents = spacy_entities_mapped(text, nlp, SPACY_TO_RESTAURANT)

    tp, fp, fn, wrong = compare_gold_vs_pred(gold_ents, pred_ents)

    all_tp.extend([(text, *x) for x in tp])
    all_fp.extend([(text, *x) for x in fp])
    all_fn.extend([(text, *x) for x in fn])
    all_wrong.extend([(text, x["text"], x["gold_label"], x["pred_label"]) for x in wrong])

    if tp or fp or fn or wrong:
        error_examples.append({
            "text": text,
            "gold_entities": gold_ents,
            "predicted_entities": pred_ents,
            "tp_count": len(tp),
            "fp_count": len(fp),
            "fn_count": len(fn),
            "wrong_label_count": len(wrong)
        })

tp_n, fp_n, fn_n, precision, recall, f1 = calc_metrics(all_tp, all_fp, all_fn, all_wrong)

print("=== Pretrained spaCy Error Analysis (Mapped Labels) ===")
print(f"True Positives: {tp_n}")
print(f"False Positives: {fp_n}")
print(f"False Negatives: {fn_n}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

=== Pretrained spaCy Error Analysis (Mapped Labels) ===
True Positives: 1406
False Positives: 1317
False Negatives: 12351
Precision: 0.5163
Recall: 0.1022
F1 Score: 0.1706


In [118]:
#Display the number of errors in each category

tp_by_label = Counter([label for _, _, _, _, label in all_tp])
fn_by_label = Counter([label for _, _, _, _, label in all_fn])
fp_by_label = Counter([label for _, _, _, _, label in all_fp])
wrong_gold = Counter([gold for _, _, gold, pred in all_wrong])
wrong_pred = Counter([pred for _, _, gold, pred in all_wrong])

print("True Positives by label")
display(pd.DataFrame(tp_by_label.items(), columns=["Label", "TP Count"]).sort_values("TP Count", ascending=False))

print("False Negatives by gold label")
display(pd.DataFrame(fn_by_label.items(), columns=["Gold Label", "FN Count"]).sort_values("FN Count", ascending=False))

print("False Positives by predicted label")
display(pd.DataFrame(fp_by_label.items(), columns=["Predicted Label", "FP Count"]).sort_values("FP Count", ascending=False))

print("Wrong-label counts by gold label")
display(pd.DataFrame(wrong_gold.items(), columns=["Gold Label", "Wrong Label Count"]).sort_values("Wrong Label Count", ascending=False))

print("Wrong-label counts by predicted label")
display(pd.DataFrame(wrong_pred.items(), columns=["Predicted Label", "Wrong Label Count"]).sort_values("Wrong Label Count", ascending=False))

True Positives by label


,Label,TP Count
0,Cuisine,1137
1,Hours,112
2,Location,99
3,Restaurant_Name,48
5,Price,8
4,Rating,2


False Negatives by gold label


,Gold Label,FN Count
1,Location,3224
3,Amenity,2245
2,Restaurant_Name,1690
4,Cuisine,1387
7,Dish,1344
5,Rating,985
6,Hours,758
0,Price,636


False Positives by predicted label


,Predicted Label,FP Count
3,Hours,481
2,Rating,353
1,Cuisine,163
4,Location,160
0,Restaurant_Name,62
5,Price,16


Wrong-label counts by gold label


,Gold Label,Wrong Label Count
0,Location,32
1,Restaurant_Name,17
3,Price,11
5,Dish,9
4,Cuisine,8
2,Amenity,4
6,Hours,1


Wrong-label counts by predicted label


,Predicted Label,Wrong Label Count
0,Restaurant_Name,30
1,Location,18
4,Hours,16
2,Cuisine,10
3,Rating,8


In [119]:
#Show samples of errors
errors_df = pd.DataFrame(error_examples)
errors_df = errors_df.sort_values(
    by=["wrong_label_count", "fn_count", "fp_count", "tp_count"],
    ascending=False
)

display(errors_df.head(15))

,text,gold_entities,predicted_entities,tp_count,fp_count,fn_count,wrong_label_count
2755,id like you to make me a reservation for two a...,"[(51, 58, upscale, Amenity), (70, 77, entrees,...","[(41, 44, two, Rating), (78, 83, 40 60, Rating)]",0,1,4,1
5220,what type of family restaurants with no bar an...,"[(13, 19, family, Amenity), (37, 43, no bar, A...","[(70, 80, 10 minutes, Hours)]",0,0,4,1
563,does downtown giovannis on east springfield st...,"[(5, 13, downtown, Location), (14, 23, giovann...","[(14, 23, giovannis, Cuisine), (27, 43, east s...",0,1,3,1
1653,how early is the dunkin donuts on ridge avenue...,"[(4, 9, early, Hours), (17, 30, dunkin donuts,...","[(34, 46, ridge avenue, Restaurant_Name), (52,...",0,1,3,1
3386,is there a new asia on james avenue with byob,"[(11, 14, new, Amenity), (15, 19, asia, Cuisin...","[(15, 19, asia, Location)]",0,0,3,1
3462,is there a restaurant in chinatown with a grea...,"[(25, 34, chinatown, Location), (42, 52, great...","[(63, 67, poek, Location)]",0,0,3,1
4245,please help me find a breakfast buffet for und...,"[(22, 31, breakfast, Hours), (32, 38, buffet, ...","[(43, 51, under 10, Hours)]",0,0,3,1
5204,what time does wendys in san lorenzo open,"[(5, 9, time, Hours), (15, 21, wendys, Restaur...","[(25, 36, san lorenzo, Location)]",0,0,3,1
5289,whats the best place for dinner within 5 miles...,"[(10, 14, best, Rating), (25, 31, dinner, Cuis...","[(59, 67, under 20, Hours)]",0,0,3,1
50,can you give me a list of the four star seafoo...,"[(30, 39, four star, Rating), (40, 47, seafood...","[(30, 34, four, Rating), (58, 67, annapolis, R...",0,1,2,1


# CONCLUSION

Looking at the initial results, if we did a straight up entity analysis, we would have an extremely low or 0.0000 F1 score. This is because the custom labels will not match up with the default spacy labels. Therefore, it can't be compared effectively. The way we tried to get around it is to create a dictionary that matched our custom library as best as possible. Even with this it is impossible to get an accurate performance score because our classification may not match 100% to what the spacy definitions are.

Based on these results, we came to the conclusion that it is not always beneficial to apply 